# Thí nghiệm chèn thuật ngữ (tuần 1, người A)

Notebook này trả lời hai câu hỏi của mốc 22/11:

1. **Chèn thuật ngữ có đủ không, hay phải tinh chỉnh LoRA?** Dịch lại các đoạn có thuật ngữ trong bộ thử bằng bản 4B
   hai lần: không chèn và có chèn (`terms.prepare` → mô hình → `terms.finalize`), rồi so tỷ lệ sai thuật ngữ
   với mốc 74% ở sách thuật toán.
2. **Mô hình có giữ nguyên dạng "overfitting (quá khớp)" và mã giữ chỗ `[T1]` không?** Hai phép thử phụ:
   cho mọi thuật ngữ ở dạng lần đầu, và thử thêm ba dạng mã giữ chỗ khác.

Cách chấm giống hệt lúc lập mốc (`eval/cham_thuat_ngu.py`, đã đối chiếu với sheet `tong_hop`).

## Chuẩn bị (làm một lần)

1. Như notebook `thu-translategemma`: chấp nhận điều khoản Gemma trên Hugging Face, thêm secret `HF_TOKEN`,
   **Accelerator: GPU T4 x2**, bật **Internet**.
2. **Đưa code lên:** nén hai thư mục `terms/` và `eval/` của repo thành một file zip, vào
   **Add Input → Upload → New Dataset** và tải file zip đó lên. Mỗi lần sửa `terms/core.py` hay bảng thuật ngữ
   thì tải lại bản mới (New Version của dataset).

Chạy thử với `QUICK_TEST = True` (20 đoạn, khoảng 5 phút). Ổn thì đổi thành `False` rồi
**Save Version → Save & Run All** (khoảng 1 giờ trên T4). Kết quả dịch được lưu dần vào cache,
phiên bị ngắt thì chạy lại sẽ dịch tiếp chỗ còn thiếu.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sacrebleu openpyxl

## 1. Cấu hình

In [ ]:
# ---- SUA O DAY ---------------------------------------------------------
QUICK_TEST = True            # True: 20 doan (10 moi sach) de kiem tra ky thuat
MODEL_ID = "google/translategemma-4b-it"
BATCH_SIZE = 4               # giam xuong 2 neu bao het bo nho GPU
SO_DOAN_THU_LAN_DAU = 100    # phep thu giu dang "overfitting (quá khớp)"
SO_DOAN_THU_MA = 40          # phep thu dang ma giu cho khac, tren cung cac doan
DANG_MA_KHAC = ["{{T{}}}", "<T{}>", "TERM{}"]   # so voi dang dang dung [T1]
INPUT_DIR = "/kaggle/input"
OUT_DIR = "/kaggle/working"
SEED = 0
# -------------------------------------------------------------------------

## 2. Nạp code và chọn đoạn

Lấy mọi đoạn văn xuôi có ít nhất một trong 20 thuật ngữ theo dõi của mốc xuất phát (270 đoạn, phủ đủ
317 + 111 lần đếm của mốc). Các trang liền nhau (cách nhau tối đa 2 trang) coi là một chương, vì
`prepare` chỉ ghi kèm ngoặc ở lần đầu mỗi chương.

In [ ]:
import glob, json, os, random, re, sys
import pandas as pd

core = glob.glob(f"{INPUT_DIR}/**/terms/core.py", recursive=True)
xlsx = glob.glob(f"{INPUT_DIR}/**/so_sanh_ban_dich_danh_gia.xlsx", recursive=True)
if len(core) != 1 or len(xlsx) != 1:
    raise SystemExit(f"Can dung 1 thu muc terms/ va 1 file so_sanh_ban_dich_danh_gia.xlsx; thay: {core} {xlsx}")
CODE_DIR = os.path.dirname(os.path.dirname(core[0]))
sys.path[:0] = [CODE_DIR, os.path.join(CODE_DIR, "eval")]

import terms
import cham_thuat_ngu as C
from terms.core import _placeholder_regex

G = {t.source: t for t in terms.load_glossary()}
print(f"Code: {CODE_DIR} | bang thuat ngu: {len(G)} muc")

data = C.doc_bang(xlsx[0])
seg = data[(data.loai_doan == "văn xuôi") & data.source.apply(lambda s: bool(C.tim_thuat_ngu(s)))]
seg = seg.sort_values("id").reset_index(drop=True)
seg["chuong"] = (seg.groupby("book").page.diff().fillna(99).abs() > 2).cumsum()
if QUICK_TEST:
    seg = seg.groupby("book").head(10).reset_index(drop=True)
print(f"{len(seg)} doan, {seg.chuong.nunique()} chuong")
seg.groupby("book").agg(doan=("id", "count"), chuong=("chuong", "nunique"))

## 3. Chuẩn bị đầu vào cho từng nhánh

- `khong_chen`: câu gốc.
- `chen`: qua `terms.prepare`, giữ trạng thái theo chương như sản phẩm thật.
- `thu_lan_dau`: mỗi đoạn dùng trạng thái mới, nên mọi thuật ngữ đều ở dạng lần đầu ("overfitting (quá khớp)",
  "ngăn xếp (stack)"). Chỉ lấy đoạn có ít nhất một dạng có ngoặc.
- `thu_ma ...`: lấy đầu vào nhánh `chen` có mã giữ chỗ, đổi `[T1]` sang dạng khác.

In [ ]:
rows = []
for _, g in seg.groupby("chuong", sort=True):
    state = terms.new_chapter()
    for r in g.itertuples():
        rows.append({"nhanh": "khong_chen", "id": r.id, "text_in": r.source})
        text_in, info = terms.prepare(r.source, state)
        rows.append({"nhanh": "chen", "id": r.id, "text_in": text_in, "info": info})


def dang_co_ngoac(found):
    # Dang lan dau co phan tieng Viet va ngoac, duoc chen thang vao cau (khong qua ma giu cho)
    return [G[t].lan_dau for t in dict.fromkeys(found) if "(" in G[t].lan_dau and not G[t].lan_dau.isascii()]


cands = []
for r in seg.itertuples():
    text_in, info = terms.prepare(r.source, terms.new_chapter())
    forms = dang_co_ngoac(info["terms"])
    if forms:
        cands.append({"nhanh": "thu_lan_dau", "id": r.id, "text_in": text_in, "info": info, "forms": forms})
random.Random(SEED).shuffle(cands)
rows += cands[:SO_DOAN_THU_LAN_DAU]

with_ph = [x for x in rows if x["nhanh"] == "chen" and x["info"]["placeholders"]]
random.Random(SEED).shuffle(with_ph)
for x in with_ph[:SO_DOAN_THU_MA]:
    x["thu_ma"] = True                       # de so dang [T1] tren cung tap doan
    for fmt in DANG_MA_KHAC:
        text, codes = x["text_in"], []
        for code in x["info"]["placeholders"]:
            alt = fmt.format(re.fullmatch(r"\[T(\d+)\]", code).group(1))
            text = text.replace(code, alt)
            codes.append(alt)
        rows.append({"nhanh": "thu_ma " + fmt.format("n"), "id": x["id"], "text_in": text, "ma": codes})

print(pd.Series([x["nhanh"] for x in rows]).value_counts().to_string())
ex = next(x for x in rows if x["nhanh"] == "chen" and x["info"]["placeholders"])
print("\nVi du dau vao nhanh chen:\n", ex["text_in"][:600], "\n", ex["info"]["placeholders"])

## 4. Dịch

Giống notebook `thu-translategemma`: nén 4-bit bitsandbytes, khuôn câu lệnh của TranslateGemma, không lấy mẫu
(`do_sample=False`). Câu trùng nhau giữa các nhánh chỉ dịch một lần.

In [ ]:
import gc, time, torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
CACHE = os.path.join(OUT_DIR, "cache_dich.json")


def load_model(model_id):
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.bfloat16)
    processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
    processor.tokenizer.padding_side = "left"
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, device_map="auto", quantization_config=quant, token=HF_TOKEN)
    model.eval()
    return processor, model


def build_prompt(processor, text):
    messages = [{"role": "user", "content": [{
        "type": "text", "source_lang_code": "en", "target_lang_code": "vi", "text": text}]}]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def translate_batch(processor, model, texts):
    prompts = [build_prompt(processor, t) for t in texts]
    enc = processor.tokenizer(prompts, return_tensors="pt", padding=True,
                              add_special_tokens=False).to(model.device)
    limit = min(1024, int(enc["input_ids"].shape[1] * 2) + 64)
    gen = model.generate(**enc, max_new_tokens=limit, do_sample=False)
    gen = gen[:, enc["input_ids"].shape[1]:]
    n_tokens = int((gen != processor.tokenizer.pad_token_id).sum())
    return [processor.tokenizer.decode(ids, skip_special_tokens=True).strip() for ids in gen], n_tokens


cache = {}
if os.path.exists(CACHE):
    with open(CACHE, encoding="utf-8") as f:
        cache = json.load(f)
texts = sorted({x["text_in"] for x in rows}, key=len, reverse=True)   # dai truoc: lo dong deu nhau
todo = [t for t in texts if t not in cache]
print(f"{len(texts)} cau can dich, {len(texts) - len(todo)} da co trong cache")

if todo:
    processor, model = load_model(MODEL_ID)
    started, new_tokens = time.time(), 0
    try:
        for start in range(0, len(todo), BATCH_SIZE):
            batch = todo[start:start + BATCH_SIZE]
            outs, n = translate_batch(processor, model, batch)
            cache.update(zip(batch, outs))
            new_tokens += n
            done = start + len(batch)
            if (start // BATCH_SIZE) % 10 == 0 or done == len(todo):
                with open(CACHE, "w", encoding="utf-8") as f:
                    json.dump(cache, f, ensure_ascii=False)
                print(f"  {done}/{len(todo)} cau, {(time.time() - started) / 60:.1f} phut", flush=True)
    finally:
        with open(CACHE, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False)
        processor = model = None
        gc.collect()
        torch.cuda.empty_cache()
    print(f"Toc do: {new_tokens / (time.time() - started):.1f} token/giay")

for x in rows:
    x["raw"] = cache[x["text_in"]]

## 5. Chấm

**Thuật ngữ** (`eval/cham_thuat_ngu.py`): mỗi cặp (đoạn, thuật ngữ theo dõi) xếp vào một loại: `bảng` (đúng cách viết
trong bảng), `đúng` (đúng nghĩa, khác cách viết), `sai` (sai nghĩa) hoặc `khác` (không thấy biến thể nào, thường là
bỏ sót). Cột `tg_4b_cu` là bản dịch lúc lập mốc, để kiểm tra lần chạy lại `khong_chen` có khớp mốc không.

Để so với mốc: `d2l` lấy `sai_nghia`, `dsa` lấy `sai_tu_dong` (= sai nghĩa + bỏ sót), đúng như cách đã đếm 6/317 và 82/111.

In [ ]:
import sacrebleu

khong = {x["id"]: x for x in rows if x["nhanh"] == "khong_chen"}
chen = {x["id"]: x for x in rows if x["nhanh"] == "chen"}

res = seg[["id", "book", "page", "chuong", "loai_doan", "source"]].copy()
res["tg_4b_cu"] = seg["tg_4b"].values
res["khong_chen"] = res.id.map(lambda i: khong[i]["raw"])
res["dau_vao_chen"] = res.id.map(lambda i: chen[i]["text_in"])
res["chen_tho"] = res.id.map(lambda i: chen[i]["raw"])
res["chen"] = res.id.map(lambda i: terms.finalize(chen[i]["raw"], chen[i]["info"]))
res["chatgpt"] = seg["chatgpt"].values

COLS = ["tg_4b_cu", "khong_chen", "chen", "chatgpt"]
tong_ket = pd.concat([C.tong_ket(res, c) for c in COLS], ignore_index=True)
tong_ket

In [ ]:
# Bien the cua tung thuat ngu, giong sheet tong_hop
bien_the = pd.DataFrame({c: C.bang_bien_the(res, c) for c in COLS})
bien_the

**chrF** so với bản ChatGPT, trung bình theo đoạn như lúc lập mốc. Chèn thuật ngữ không được làm câu kém đi rõ rệt.

In [ ]:
def chrf(col):
    s = res.apply(lambda r: sacrebleu.sentence_chrf(str(r[col]), [str(r["chatgpt"])]).score, axis=1)
    return s.groupby(res.book).mean().round(1)


chrf_tab = pd.DataFrame({c: chrf(c) for c in ["tg_4b_cu", "khong_chen", "chen"]})
chrf_tab

**Mã giữ chỗ** trong nhánh `chen`: còn nguyên `[T1]`, hay chỉ khớp sau khi nới lỏng (đổi ngoặc, thêm dấu cách,
`finalize` vẫn đặt lại được), hay mất hẳn (từ tiếng Anh bị mất khỏi bản dịch).

In [ ]:
ph = pd.DataFrame([
    {"id": x["id"], "ma": code, "tu": original,
     "giu_nguyen": code in x["raw"],
     "khop_noi_long": bool(_placeholder_regex(code).search(x["raw"]))}
    for x in chen.values() for code, original in x["info"]["placeholders"].items()
], columns=["id", "ma", "tu", "giu_nguyen", "khop_noi_long"])
con_sot = res.chen.str.contains(r"\[T\d+\]").sum()
print(f"{len(ph)} ma giu cho: giu nguyen {ph.giu_nguyen.mean():.1%}, "
      f"finalize dat lai duoc {ph.khop_noi_long.mean():.1%}, mat {(~ph.khop_noi_long).sum()}")
print(f"Ban dich cuoi con sot ma [Tn]: {con_sot} doan")
ph[~ph.khop_noi_long].head(20)

**Dạng lần đầu** ("overfitting (quá khớp)", "ngăn xếp (stack)") trong phép thử `thu_lan_dau`.

In [ ]:
def kiem_dang(form, out):
    f, o = form.lower(), str(out).lower()
    a, b = re.fullmatch(r"(.+?) \((.+)\)", f).groups()
    if f in o:
        return "giữ nguyên"
    if f"{b} ({a})" in o:
        return "đảo thứ tự"
    if a in o and b in o:
        return "đủ hai phần, khác dạng"
    if a in o or b in o:
        return "chỉ còn một phần"
    return "mất"


lan_dau = pd.DataFrame([
    {"id": x["id"], "dang": f, "ket_qua": kiem_dang(f, x["raw"]), "dau_vao": x["text_in"], "ban_dich": x["raw"]}
    for x in rows if x["nhanh"] == "thu_lan_dau" for f in x["forms"]
], columns=["id", "dang", "ket_qua", "dau_vao", "ban_dich"])
print(lan_dau.ket_qua.value_counts(normalize=True).round(3).to_string())
lan_dau[lan_dau.ket_qua != "giữ nguyên"][["dang", "ket_qua", "ban_dich"]].head(15)

**Các dạng mã giữ chỗ khác**, trên cùng các đoạn: tỷ lệ còn nguyên văn trong bản dịch.

In [ ]:
dang_ma = [{"dang": x["nhanh"][len("thu_ma "):], "giu_nguyen": m in x["raw"]}
           for x in rows if x["nhanh"].startswith("thu_ma ") for m in x["ma"]]
dang_ma += [{"dang": "[Tn] (đang dùng)", "giu_nguyen": code in x["raw"]}
            for x in chen.values() if x.get("thu_ma") for code in x["info"]["placeholders"]]
dang_ma = pd.DataFrame(dang_ma, columns=["dang", "giu_nguyen"]).groupby("dang").giu_nguyen.agg(["mean", "count"])
dang_ma

## 6. Xuất kết quả

File `thi_nghiem_chen_thuat_ngu.xlsx` trong tab Output: sheet `so_sanh` đặt các bản dịch cạnh nhau để đọc tay,
cùng các sheet số liệu. Ô cuối in sẵn các dòng để dán vào `eval/ket_qua.csv`.

In [ ]:
from datetime import date
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from openpyxl.styles import Alignment

path = os.path.join(OUT_DIR, "thi_nghiem_chen_thuat_ngu.xlsx")
clean = lambda d: d.map(lambda v: ILLEGAL_CHARACTERS_RE.sub("", v) if isinstance(v, str) else v)
with pd.ExcelWriter(path, engine="openpyxl") as w:
    clean(res).to_excel(w, sheet_name="so_sanh", index=False)
    ws = w.sheets["so_sanh"]
    for col in "FGHIJKL":
        ws.column_dimensions[col].width = 55
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "G2"
    tong_ket.to_excel(w, sheet_name="tong_ket", index=False)
    bien_the.to_excel(w, sheet_name="bien_the")
    chrf_tab.to_excel(w, sheet_name="chrF")
    ph.to_excel(w, sheet_name="ma_giu_cho", index=False)
    clean(lan_dau).to_excel(w, sheet_name="lan_dau", index=False)
    dang_ma.to_excel(w, sheet_name="dang_ma")
print("Da luu:", path)


def dong_ket_qua(col, cau_hinh):
    t = C.tong_ket(res, col).set_index("bo")
    get = lambda bo, k: t[k].get(bo, 0)
    return {
        "ngay": date.today().isoformat(), "he_thong": "TranslateGemma 4B",
        "bo_thu": f"{len(res)} doan co thuat ngu (d2l + CLRS)", "cau_hinh": cau_hinh,
        "chrF_d2l": chrf_tab[col].get("d2l-en", ""), "chrF_dsa": chrf_tab[col].get("DSA", ""),
        "sai_thuat_ngu_d2l": f'{get("d2l", "sai_nghia")}/{get("d2l", "so_cap")}',
        "sai_thuat_ngu_dsa": f'{get("dsa", "sai_tu_dong")}/{get("dsa", "so_cap")}',
        "nhat_quan_thuat_ngu": f'{t.loc["tất cả", "nhat_quan"]:.0f}%', "giu_nguyen_code": "",
        "ghi_chu": "cham tu dong eval/cham_thuat_ngu.py" + (" - QUICK_TEST" if QUICK_TEST else ""),
    }


cfg = "Kaggle T4, nen 4-bit bitsandbytes, "
lines = pd.DataFrame([dong_ket_qua("khong_chen", cfg + "khong chen thuat ngu (chay lai)"),
                      dong_ket_qua("chen", cfg + f"chen thuat ngu (terms.prepare/finalize, ma {terms.core.PLACEHOLDER})")])
print(lines.to_csv(index=False, header=False))

## Đọc kết quả thế nào

1. **Lần chạy lại có khớp mốc không:** dòng `khong_chen` trong `tong_ket` phải gần dòng `tg_4b_cu`
   (cùng mô hình, cùng cấu hình). Lệch nhiều thì xem lại môi trường trước khi kết luận.
2. **Chèn thuật ngữ giúp được bao nhiêu:** so `chen` với `khong_chen` ở bộ `dsa` (mốc 74%) và cột `nhat_quan` (mốc 78%, ChatGPT 98%).
3. **Có làm hỏng câu không:** chrF của `chen` không nên thấp hơn `khong_chen` quá vài điểm. Đọc tay khoảng 20 đoạn ở sheet
   `so_sanh` (cột `dau_vao_chen` và `chen`), chú ý cụm ghép như "memory cell internal state" và từ dùng như động từ
   ("upper-bound", "designs feature").
4. **Mã giữ chỗ và dạng lần đầu:** nếu `[T1]` mất nhiều hơn vài phần trăm thì đổi `PLACEHOLDER` trong `terms/core.py` sang
   dạng tốt nhất ở bảng `dang_ma`. Nếu dạng lần đầu hay bị đảo hoặc mất thì cân nhắc bảo vệ cả dạng đó bằng mã giữ chỗ.

**Gợi ý ngưỡng cho mốc 22/11** (cả nhóm chốt lại): không cần LoRA nếu với nhánh `chen` thì sai thuật ngữ ở `dsa` dưới
khoảng 15%, nhất quán từ 90% trở lên, chrF không giảm quá 2 điểm, và mã giữ chỗ được đặt lại từ 98% trở lên.
Thiếu điều nào thì LoRA (tuần 2) nhắm vào đúng điểm đó.